In [1]:
import pandas as pd
from datasets import Dataset
import torch

# Load dataset
df = pd.read_csv("news_data_1M_lines.txt", sep="\t", header=None, on_bad_lines="skip").iloc[60000:360000]
df.columns = ["incorrect", "correct"]
df = df.dropna().drop_duplicates()
df["incorrect"] = df["incorrect"].str.strip()
df["correct"] = df["correct"].str.strip()

print("Samples:", len(df))
display(df.head())

dataset = Dataset.from_pandas(df)

Samples: 299893


,incorrect,correct
60000,"खरँतर, केद्र सरकरने रज्यन ेणे अससलेले मगल वर्ष...","खरंतर, केंद्र सरकारने राज्यांना देणे असलेले मा..."
60001,सथ रिोग कयद्यचं्य नवखल केँद्र सरकर तुे नकरुहि ...,साथ रोग कायद्याच्या नावाखाली केंद्र सरकार ते न...
60002,पीण केद्रने तसँ केलाल नह.,पण केंद्राने तसं केलेलं नाही.
60003,"आतचंि जि मगण होतय, त मर्चंनँतरच्य जएसटिचंि आहे","आताची जी मागणी होतेय, ती मार्चनंतरच्या जीएसटीच..."
60004,"त्यबबतह केद्रनँ मदतिचंि भुमीक घेतलय,' अस फडणवि...","त्याबाबतही केंद्रानं मदतीची भूमिका घेतलीय,' अस..."


In [2]:
import re

def clean_marathi_text(text):

    text = re.sub(r"[A-Za-z]*[0-9०-९]+", "", text)

    text = re.sub(r"[^\u0900-\u097F\s.,।'’\"]+", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["incorrect"] = df["incorrect"].apply(clean_marathi_text)
df["correct"] = df["correct"].apply(clean_marathi_text)

display(df.head())

,incorrect,correct
60000,"खरँतर, केद्र सरकरने रज्यन ेणे अससलेले मगल वर्ष...","खरंतर, केंद्र सरकारने राज्यांना देणे असलेले मा..."
60001,सथ रिोग कयद्यचं्य नवखल केँद्र सरकर तुे नकरुहि ...,साथ रोग कायद्याच्या नावाखाली केंद्र सरकार ते न...
60002,पीण केद्रने तसँ केलाल नह.,पण केंद्राने तसं केलेलं नाही.
60003,"आतचंि जि मगण होतय, त मर्चंनँतरच्य जएसटिचंि आहे","आताची जी मागणी होतेय, ती मार्चनंतरच्या जीएसटीच..."
60004,"त्यबबतह केद्रनँ मदतिचंि भुमीक घेतलय,' अस फडणवि...","त्याबाबतही केंद्रानं मदतीची भूमिका घेतलीय,' अस..."


In [3]:
print("Empty incorrect rows:", df["incorrect"].isna().sum())
print("Empty correct rows:", df["correct"].isna().sum())

df = df.dropna().reset_index(drop=True)

print(df.sample(5))

Empty incorrect rows: 0
Empty correct rows: 0
                                                incorrect  \
54948   सॅमसगचं्य य स्मर्टफोनल सप्टेँबर महन्यत लँचं कर...   
253557  मत्र यवर नयत्रण ठेवण्यसठ नगरपलकेचं सक्षषम यँत्...   
81935   वृत्तसेव, पलघरयेथिल सोनोपत दडेकर महवीद्यलयचं्य...   
6332                                        रपयवर बँद झल.   
59055   नगरकनि सयँकळि घरचं्य अगणत, बल्कनत, टेरेसव दवे ...   

                                                  correct  
54948   सॅमसंगच्या या स्मार्टफोनला सप्टेंबर महिन्यात ल...  
253557  मात्र यावर नियंत्रण ठेवण्यासाठी नगरपालिकेची सक...  
81935   वृत्तसेवा, पालघरयेथील सोनोपंत दांडेकर महाविद्य...  
6332                                   रुपयांवर बंद झाला.  
59055   नागरिकांनी सायंकाळी घराच्या अंगणात, बाल्कनीत, ...  


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))

Train samples: 269903
Validation samples: 29990


In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(train_dataset[0])

{'incorrect': 'परवे कॅमेऱ्यत कैदमँगळवरि देवळल कॅमप् येथल खडोब टेकडि परसरत य नीर्भय पथकने न जणँवर करवई केलि.', 'correct': 'पुरावे कॅमेऱ्यात कैदमंगळवारी देवळाली कॅम्प येथील खंडोबा टेकडी परिसरात या निर्भया पथकाने नऊ जणांवर कारवाई केली.', '__index_level_0__': 163768}


In [6]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

MODEL_CHECKPOINT = "ai4bharat/indicbart"  
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["incorrect"], text_target=batch["correct"],
                     truncation=True, padding="max_length", max_length=64)

# Apply tokenization
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# Remove unnecessary columns
train_dataset = train_dataset.remove_columns(["incorrect", "correct", "__index_level_0__"])
val_dataset = val_dataset.remove_columns(["incorrect", "correct", "__index_level_0__"])

Map:   0%|          | 0/269903 [00:00<?, ? examples/s]

Map:   0%|          | 0/29990 [00:00<?, ? examples/s]

In [7]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [8]:
print(train_dataset[0]['labels'])
print(sum([1 for i in train_dataset[0]['labels'] if i != tokenizer.pad_token_id]))

[2, 49346, 7879, 335, 6071, 18, 11796, 50620, 8948, 29, 3634, 2586, 52162, 2963, 5614, 12748, 5871, 294, 10673, 116, 42528, 42712, 32368, 1788, 4377, 1828, 748, 6, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
29


In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=False,  
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

r
trainer.train()
model.save_pretrained("lang_analyzer_300k_model")
tokenizer.save_pretrained("lang_analyzer_300k_model")

C:\Users\samar\AppData\Local\Temp\ipykernel_18204\986872969.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.479200,0.383320
2,0.373900,0.295207
3,0.324900,0.259151
4,0.290600,0.242393
5,0.286800,0.237363


('lang_analyzer_300k_model\\tokenizer_config.json',
 'lang_analyzer_300k_model\\special_tokens_map.json',
 'lang_analyzer_300k_model\\spiece.model',
 'lang_analyzer_300k_model\\added_tokens.json',
 'lang_analyzer_300k_model\\tokenizer.json')

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load trained model + tokenizer
MODEL_DIR = "lang_analyzer_model"   # your saved model folder
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # ⚡ set to evaluation mode for speed

# --- Function for correcting a single sentence ---
def correct_sentence(incorrect_sentence, max_length=64):
    # Tokenize
    inputs = tokenizer(
        incorrect_sentence,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    # 🚫 Remove token_type_ids (MBart/IndicBART doesn’t use them)
    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]

    # Send to GPU/CPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Fast greedy decoding (⚡ <1 sec)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=8,       # greedy search (faster than beam search)
        do_sample=False
    )

    # Decode prediction
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# --- Example ---
incorrect_sentence = "य प्रकरणि न्ययलयत दोषरोपपत्र दखल करण्यत आआले आहेु."
corrected = correct_sentence(incorrect_sentence)

print("Incorrect:", incorrect_sentence)
print("Corrected:", corrected)

Incorrect: य प्रकरणि न्ययलयत दोषरोपपत्र दखल करण्यत आआले आहेु.
Corrected: या प्रकरणी न्यायालयात दोषारोपपत्र दाखल करण्यात आले आहे.


In [ ]:
from evaluate import load
import torch
from tqdm import tqdm  # progress bar

sacrebleu = load("sacrebleu")

model.eval()
predictions = []
references = []

max_length = 64

# Loop with progress bar
for example in tqdm(val_dataset, desc="Generating predictions"):
    # Convert input_ids and attention_mask to tensors
    input_ids = torch.tensor(example["input_ids"], dtype=torch.long).unsqueeze(0).to(device)
    attention_mask = torch.tensor(example["attention_mask"], dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            num_beams=8
        )

    # Decode prediction
    corrected_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(corrected_sentence)

    # Prepare reference sentence
    labels = [id for id in example["labels"] if id != -100]
    reference_sentence = tokenizer.decode(labels, skip_special_tokens=True)
    references.append([reference_sentence])  # sacrebleu expects list of lists

# Compute BLEU score
results = sacrebleu.compute(predictions=predictions, references=references)
print("SacreBLEU score:", results["score"])

Using the latest cached version of the module from C:\Users\samar\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--sacrebleu\28676bf65b4f88b276df566e48e603732d0b4afd237603ebdf92acaacf5be99b (last modified on Thu Oct  2 01:28:38 2025) since it couldn't be found locally at evaluate-metric--sacrebleu, or remotely on the Hugging Face Hub.
Generating predictions:   0%|                                                     | 33/29990 [00:23<9:23:38,  1.13s/it]